In [3]:
import pandas as pd
import ast

# -----------------------------
# 1. Load core files
# -----------------------------
# Features and Echonest have 3 metadata rows at the top → skip them
features = pd.read_csv("data/fma-small/features.csv", skiprows=[0,1,2], low_memory=False)

# Load raw echonest file without skipping
echonest_raw = pd.read_csv("data/fma-small/echonest.csv", low_memory=False)

# Row 1 contains the real column names → set them as header
echonest_raw.columns = echonest_raw.iloc[1]

# Drop the first two metadata rows (row 0 and row 1)
echonest = echonest_raw.drop([0,1])

# Rename first column to track_id
echonest.rename(columns={echonest.columns[0]: "track_id"}, inplace=True)

# Tracks has a multi-index header (two rows) → load with header=[0,1]
tracks = pd.read_csv("data/fma-small/tracks.csv", header=[0,1], low_memory=False)

# Flatten multi-index headers
tracks.columns = ['_'.join(col).strip() for col in tracks.columns.values]

# The first column is messy → force it to be 'track_id'
tracks.rename(columns={tracks.columns[0]: "track_id"}, inplace=True)

# Drop the first row (it only contains the string 'track_id' and NaNs)
tracks = tracks.drop(index=0)

# Genres is a simple lookup table (genre_id → title, parent)
genres = pd.read_csv("data/fma-small/genres.csv", low_memory=False)

# To avoid surprises, normalize all IDs right after loading
for df in [tracks, features, echonest]:
    df["track_id"] = pd.to_numeric(df["track_id"], errors="coerce")

# Rename the messy first column to track_id
tracks.rename(columns={tracks.columns[0]: "track_id"}, inplace=True)

# -----------------------------
# 3. Parse multi-genre column
# -----------------------------
# track_genres_all is stored as a stringified list → convert to Python list
tracks["track_genres_all"] = tracks["track_genres_all"].apply(
    lambda x: ast.literal_eval(x) if pd.notnull(x) else []
)

# -----------------------------
# 4. Explode genres into rows
# -----------------------------
# Each track with multiple genres becomes multiple rows
tracks_exploded = tracks.explode("track_genres_all")

# -----------------------------
# 5. Merge with genres.csv
# -----------------------------
# Map genre IDs to human-readable titles
tracks_genres_named = tracks_exploded.merge(
    genres, left_on="track_genres_all", right_on="genre_id", how="left"
)

# -----------------------------
# 6. Group genres back into lists
# -----------------------------
# Aggregate all genre titles per track into a list
tracks_grouped = (
    tracks_genres_named.groupby("track_id")["title"]
    .apply(list)
    .reset_index()
    .rename(columns={"title": "genre_list"})
)

# -----------------------------
# 7. Merge genre list back into core tracks
# -----------------------------
tracks_named = tracks.merge(tracks_grouped, on="track_id", how="left")

# -----------------------------
# 8. Merge with features and echonest
# -----------------------------
songs_enriched = tracks_named.merge(features, left_on="track_id", right_on="track_id", how="left")
songs_enriched = songs_enriched.merge(echonest, on="track_id", how="left")

# --- Safeguard: coalesce artist_name_x and artist_name_y ---
songs_enriched["artist_name"] = songs_enriched["artist_name_x"].fillna(songs_enriched["artist_name_y"])

# If both are missing, fill with a placeholder
songs_enriched["artist_name"] = songs_enriched["artist_name"].fillna("Unknown Artist")

# Drop duplicates
songs_enriched.drop(columns=["artist_name_x", "artist_name_y"], inplace=True)

# --- Keep the required params ---
songs_final = songs_enriched[[
    "track_id",
    "track_title",
    "artist_name",
    "album_title",
    "genre_list"
]]

# -----------------------------
# ✅ Final DataFrame: songs_enriched
# -----------------------------
# Contains:
# - track_track_id (primary key)
# - album_title, artist_name, track_genre_top
# - genre_list (all human-readable genres)
# - 518 numeric audio features
# - Echonest descriptors (acousticness, danceability, energy, etc.)


In [2]:
# Collect all unique genres into a set
all_genres = set()

for val in songs_final["genre_list"].dropna():
    if isinstance(val, list):  # case: already a list of genres
        for g in val:
            all_genres.add(str(g).strip())
    else:  # case: single string or other type
        all_genres.add(str(val).strip())

print(sorted(all_genres)[:50])   # preview first 50 genres
print("Total distinct genres:", len(all_genres))


NameError: name 'songs_final' is not defined

In [4]:
# Total rows
total_rows = len(songs_final)

# Count missing and non-missing per column
missing_count = songs_final.isnull().sum()
non_missing_count = total_rows - missing_count
missing_percent = (missing_count / total_rows) * 100

# Combine into one DataFrame
missing_summary = pd.DataFrame({
    "non_missing": non_missing_count,
    "missing": missing_count,
    "missing_percent": missing_percent.round(2)
})

print(missing_summary)


             non_missing  missing  missing_percent
track_id          106574        0             0.00
track_title       106573        1             0.00
artist_name       106574        0             0.00
album_title       105549     1025             0.96
genre_list        106574        0             0.00
